# Lab 5: Deploy a Google Search Agent to Agent Platform (Agent Runtime)

This notebook builds a deliberately simple agent and then **deploys it** as a managed service on Google Cloud, so the
focus is on the deployment steps rather than on the agent itself.

| Step | What happens |
|---|---|
| 1. Agent | An ADK agent with one tool, ADK's built-in **Google Search** tool |
| 2. Wrap | `AdkApp` wraps the agent so Agent Platform can run it as a service |
| 3. Local test | Query the wrapped agent inside this notebook (in-memory sessions) |
| 4. Deploy | `client.agent_engines.create(...)` stages the agent in a Cloud Storage bucket and creates a managed deployment (a `reasoningEngines` resource). This takes several minutes |
| 5. Remote test | Query the **deployed** agent and print the events it returns, including the Google searches it ran |
| 6. Reconnect | Re-open the deployment by its resource name, which shows it lives in the cloud and not in this notebook |
| 7. Cleanup | Delete the deployment (optional, switched off by default) |

**Naming.** Google now calls this service *Agent Runtime* in *Gemini Enterprise Agent Platform* (earlier: Vertex AI Agent
Engine). The Python SDK still uses the older names: `vertexai.Client`, `client.agent_engines`, and `AdkApp`.

**How to run:** run the cells from top to bottom. After the install cell finishes, restart the session (Runtime menu)
and continue from the configuration cell. Deployment (section 7) takes several minutes; let it finish.

## 1. Install dependencies

In [ ]:
%pip install --quiet --upgrade "google-cloud-aiplatform[agent_engines,adk]>=1.112"

Restart the session now (**Runtime > Restart session**), then continue with the next cell.
Warnings about dependency conflicts from the install are safe to ignore.

In [ ]:
import importlib.metadata as metadata

from packaging.version import Version

MIN_AIPLATFORM = "1.112"   # first google-cloud-aiplatform release with the Agent Platform SDK used in this notebook

aiplatform_version = metadata.version("google-cloud-aiplatform")
print("google-cloud-aiplatform:", aiplatform_version, "| google-adk:", metadata.version("google-adk"))
if Version(aiplatform_version) < Version(MIN_AIPLATFORM):
    raise RuntimeError(
        f"google-cloud-aiplatform {aiplatform_version} is too old for this lab. "
        "Run the install cell above again, restart the session, and re-run this cell."
    )
print("OK: the installed versions are new enough.")

## 2. Configuration

Two settings deserve a note:

- `USE_AGENT_IDENTITY` follows the instructor's example (`identity_type=AGENT_IDENTITY`), which gives the deployed agent
  its own identity. If the deployed agent later fails with `PERMISSION_DENIED`, see the troubleshooting section at the end.
- `DELETE_AGENT_WHEN_DONE` stays `False` so the deployment is still there when your work is reviewed. Cleanup is section 10.

In [ ]:
import os

import vertexai

PROJECT_ID = "qwiklabs-gcp-02-96c7c0cbf38a"
LOCATION = "us-central1"            # region where the agent is deployed
MODEL_LOCATION = "us-central1"      # region used to call Gemini (gemini-2.5-flash worked here in the earlier labs)
GEMINI_MODEL = "gemini-2.5-flash"

# Cloud Storage bucket where the SDK stages the agent's code during deployment.
# Student accounts often cannot create buckets: if section 3 says so, paste an existing bucket here (gs://...).
STAGING_BUCKET = "gs://qwiklabs-gcp-04-99e6e44330e1-agent-staging"

USE_AGENT_IDENTITY = True           # follow the instructor's example; see troubleshooting if this causes permission errors
DELETE_AGENT_WHEN_DONE = False      # set to True to delete the deployment at the end (section 10)

# Gemini through Vertex AI, used when the agent runs inside this notebook (local test).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = MODEL_LOCATION

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Project:", PROJECT_ID)
print("Deploying to:", LOCATION, "| staging bucket:", STAGING_BUCKET)

## 3. Prepare the project

Deploying needs a few APIs enabled and a Cloud Storage bucket for staging. Both cells are safe to run more than once.
In a Qwiklabs project most APIs are already enabled, so a permission message from the first cell can usually be ignored.
If the lab account cannot create buckets, the bucket cell lists the ones that exist; copy one into `STAGING_BUCKET` and re-run.

In [ ]:
import subprocess

APIS = [
    "aiplatform.googleapis.com",           # Agent Platform
    "storage.googleapis.com",              # Cloud Storage (staging bucket)
    "logging.googleapis.com",
    "monitoring.googleapis.com",
    "cloudtrace.googleapis.com",
    "telemetry.googleapis.com",
    "cloudresourcemanager.googleapis.com",
]

try:
    result = subprocess.run(
        ["gcloud", "services", "enable", *APIS, "--project", PROJECT_ID],
        capture_output=True, text=True, timeout=300,
    )
    print("gcloud return code:", result.returncode)
    print((result.stdout + result.stderr).strip()[-500:] or "(no output)")
    if result.returncode != 0:
        print("\nNot fatal: if the APIs are already enabled in your lab project, just continue.")
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print(f"Skipped enabling APIs ({type(exc).__name__}). If deployment fails, enable them in the console.")

In [ ]:
from google.api_core import exceptions as gcp_exceptions
from google.cloud import storage

bucket_name = STAGING_BUCKET.removeprefix("gs://")
storage_client = storage.Client(project=PROJECT_ID)


def existing_buckets():
    """Names of buckets in the project, or [] if the account may not list them."""
    try:
        return [b.name for b in storage_client.list_buckets()]
    except Exception:
        return []


def stop_with_help(reason):
    names = existing_buckets()
    if names:
        listing = "Buckets in this project:\n  " + "\n  ".join(f"gs://{n}" for n in names)
    else:
        listing = "No buckets were visible to this account."
    raise RuntimeError(
        f"{reason}\n{listing}\n"
        "Set STAGING_BUCKET in the config cell (section 2) to a gs://... bucket you can write to, "
        "then run from the config cell again. If there is none, ask your instructor for one."
    ) from None


try:
    storage_client.get_bucket(bucket_name)
    print("Staging bucket is ready:", STAGING_BUCKET)
except gcp_exceptions.NotFound:
    try:
        storage_client.create_bucket(bucket_name, location="US")
        print("Created staging bucket:", STAGING_BUCKET)
    except gcp_exceptions.Forbidden:
        stop_with_help(f"This account cannot create buckets, and {STAGING_BUCKET} does not exist.")
except gcp_exceptions.Forbidden:
    stop_with_help(f"No access to {STAGING_BUCKET}.")

## 4. The agent

A deliberately simple agent: one instruction and one tool, ADK's built-in `google_search`. `google_search` runs inside
the Gemini model (Gemini searches Google and grounds its answer in the results), so there is no function code to write.

`AdkApp` wraps the agent with the query methods that Agent Platform needs (`async_stream_query` and session handling).
The same wrapped object is used for the local test and for the deployment.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from vertexai import agent_engines

AGENT_INSTRUCTIONS = """
You are a helpful research assistant. Use Google Search to find current, accurate information, then
answer the user's question in a short paragraph. Mention the source names you relied on. If the search
does not give a reliable answer, say so instead of guessing.
"""

google_search_agent = Agent(
    name="google_search_agent",
    model=GEMINI_MODEL,
    description="Answers questions by searching Google.",
    instruction=AGENT_INSTRUCTIONS,
    tools=[google_search],  # ADK built-in Google Search tool
)

app = agent_engines.AdkApp(agent=google_search_agent)
print("Created agent:", google_search_agent.name, "-> wrapped in AdkApp")

## 5. Helpers to query an agent and show its events

`async_stream_query` streams the agent's **events** as dictionaries. These helpers work on both the local `AdkApp` and the
deployed agent, because both offer the same method. Each event is turned into rows for a table: tool calls and results,
the **Google Search queries** Gemini ran (found in the event's grounding metadata), and the model's text.

`ask` also retries with increasing waits when the model reports no capacity (HTTP 429), which can happen on shared lab
projects.

In [ ]:
import asyncio
from typing import Any

from IPython.display import Markdown, display


def short(text: Any, limit: int = 200) -> str:
    """Collapses whitespace and truncates text so it fits in one table cell.

    Args:
        text: The value to shorten (converted with str()).
        limit: Maximum number of characters to keep.

    Returns:
        The text on a single line, ending in "..." if it was truncated.
    """
    flat = " ".join(str(text).split())
    return flat if len(flat) <= limit else flat[: limit - 3] + "..."


def field(data: dict[str, Any], snake: str, camel: str, default: Any = None) -> Any:
    """Reads a key that may be spelled in snake_case or camelCase.

    Args:
        data: The dictionary to read from.
        snake: The snake_case spelling of the key.
        camel: The camelCase spelling of the key.
        default: Value to return when neither spelling is present.

    Returns:
        The value stored under either spelling, or the default.
    """
    return data.get(snake, data.get(camel, default))


def event_rows(event: dict[str, Any]) -> list[tuple[str, str]]:
    """Summarizes one agent event as (kind, detail) rows for the events table.

    Args:
        event: An event dictionary returned by async_stream_query.

    Returns:
        A list of (kind, detail) pairs such as ("tool call", "name(args)"),
        ("Google Search query", "..."), or ("response", "...").
    """
    rows: list[tuple[str, str]] = []
    for part in (event.get("content") or {}).get("parts") or []:
        call = field(part, "function_call", "functionCall")
        result = field(part, "function_response", "functionResponse")
        if call:
            rows.append(("tool call", f"{call.get('name')}({short(call.get('args', {}), 100)})"))
        elif result:
            rows.append(("tool result", f"{result.get('name')} -> {short(result.get('response', {}), 100)}"))
        elif part.get("text") and not part.get("thought"):
            rows.append(("response", short(part["text"], 160)))
    grounding = field(event, "grounding_metadata", "groundingMetadata") or {}
    for query in field(grounding, "web_search_queries", "webSearchQueries") or []:
        rows.append(("Google Search query", str(query)))
    return rows


def event_text(event: dict[str, Any]) -> str:
    """Returns the plain text of an event (model thoughts are skipped).

    Args:
        event: An event dictionary returned by async_stream_query.

    Returns:
        The text parts of the event joined by spaces, or an empty string.
    """
    parts = (event.get("content") or {}).get("parts") or []
    return " ".join(p["text"] for p in parts if p.get("text") and not p.get("thought"))


async def ask(
    agent_app: Any,
    question: str,
    user_id: str = "student",
    attempts: int = 5,
    wait_s: int = 20,
) -> tuple[str, list[dict[str, Any]]]:
    """Asks an agent one question and collects its events.

    Works with the local AdkApp and with a deployed agent.

    Args:
        agent_app: An object with an async_stream_query method.
        question: The user's question.
        user_id: Identifier for the session's user.
        attempts: Maximum number of tries when the model reports no capacity.
        wait_s: Base wait in seconds between tries; the wait grows each attempt.

    Returns:
        A tuple (final_text, trace). The trace has one row per event with the
        step number, the author agent, the kind of event, and a detail string.

    Raises:
        RuntimeError: If the agent reports an error, or access is denied.
    """
    for attempt in range(1, attempts + 1):
        trace: list[dict[str, Any]] = []
        final_text = ""
        try:
            async for event in agent_app.async_stream_query(user_id=user_id, message=question):
                error = field(event, "error_message", "errorMessage")
                if error:
                    raise RuntimeError(f"{field(event, 'error_code', 'errorCode')}: {error}")
                for kind, detail in event_rows(event):
                    trace.append({
                        "step": len(trace) + 1,
                        "author": event.get("author", "?"),
                        "event": kind,
                        "detail": detail,
                    })
                text = event_text(event)
                if text:
                    final_text = text
            return final_text, trace
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            if "PERMISSION_DENIED" in message or "permission denied" in message.lower():
                raise RuntimeError(
                    "The agent was denied access (PERMISSION_DENIED). See the troubleshooting section at the "
                    f"end of this notebook.\n\nOriginal error: {short(message, 400)}"
                ) from exc
            busy = any(key in message for key in ("RESOURCE_EXHAUSTED", "429", "Quota exceeded", "quota"))
            if not busy or attempt == attempts:
                raise
            delay = wait_s * attempt
            print(f"Model busy (attempt {attempt}/{attempts}). Waiting {delay}s...")
            await asyncio.sleep(delay)
    return "", []


def show_events(title: str, question: str, final_text: str, trace: list[dict[str, Any]]) -> None:
    """Displays the events of one question as a table, followed by the final answer.

    Args:
        title: A heading describing which agent was queried.
        question: The question that was asked.
        final_text: The agent's final answer.
        trace: The event rows returned by ask.
    """
    lines = ["| Step | Agent | Event | Detail |", "|---|---|---|---|"]
    for row in trace:
        detail = row["detail"].replace("|", "\\|")
        lines.append(f"| {row['step']} | `{row['author']}` | {row['event']} | {detail} |")
    display(Markdown(
        f"### {title}\n\n**Question:** {question}\n\n" + "\n".join(lines)
        + f"\n\n**Final answer:** {final_text}"
    ))

## 6. Test locally before deploying

The wrapped agent runs right here in the notebook with in-memory sessions. If it does not work locally, it will not
work once deployed, so this is the cheap place to catch mistakes. The table lists the events in order, including the Google
searches Gemini ran.

In [ ]:
LOCAL_QUESTIONS = [
    "What is the tallest mountain in the world, and how tall is it?",
    "Who is the current Secretary-General of the United Nations?",
]

for question in LOCAL_QUESTIONS:
    answer, trace = await ask(app, question)
    show_events("Local test", question, answer, trace)
    assert answer.strip(), f"No answer for: {question}"
    await asyncio.sleep(3)  # small pause to stay under rate limits

print("Local test passed: the agent answers with Google Search.")

## 7. Deploy to Agent Platform

`client.agent_engines.create(agent=app, config={...})` does the deployment. Behind this one call the SDK packages the agent
(pickling the `AdkApp`), uploads it to the staging bucket, and creates a managed deployment together with a managed session
service. **It takes several minutes and prints progress as it goes. Do not stop the cell.**

The config follows the instructor's example:

| Setting | Meaning |
|---|---|
| `requirements` | Python packages installed in the deployed container. Pinned here to the versions installed in this notebook so the deployed code matches what was tested |
| `staging_bucket` | Where the SDK uploads the agent's code |
| `env_vars` | Environment variables for the deployed agent; `GOOGLE_CLOUD_LOCATION` sets the region used to call Gemini |
| `identity_type` | `AGENT_IDENTITY` gives the deployed agent its own identity |
| `display_name`, `description` | Labels shown in the console |

In [ ]:
import importlib.metadata as metadata
import time

from vertexai import types

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

config = {
    "requirements": [
        f"google-cloud-aiplatform[agent_engines,adk]=={metadata.version('google-cloud-aiplatform')}",
        f"cloudpickle=={metadata.version('cloudpickle')}",
    ],
    "staging_bucket": STAGING_BUCKET,
    "env_vars": {"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION},
    "display_name": "Google Search Agent",
    "description": "This agent can search Google and return answers.",
}
if USE_AGENT_IDENTITY:
    config["identity_type"] = types.IdentityType.AGENT_IDENTITY

print("Deploying... this takes several minutes.")
started = time.time()
remote_agent = client.agent_engines.create(agent=app, config=config)
minutes = (time.time() - started) / 60

RESOURCE_NAME = remote_agent.api_resource.name
with open("deployed_agent.txt", "w") as resource_file:
    resource_file.write(RESOURCE_NAME)

print(f"\nDeployed in {minutes:.1f} minutes.")
print("Display name :", remote_agent.api_resource.display_name)
print("Resource name:", RESOURCE_NAME)

You can also see the deployment in the console: use the search bar at the top and look for **Agent Runtime** (or
**Deployments** under Agent Platform). The resource name above is its unique ID and is also saved in `deployed_agent.txt`.

## 8. Test the deployed agent

Now the same kind of question is answered by the **deployed** agent, running in Google Cloud instead of in this notebook.
`remote_agent.async_stream_query` works exactly like the local `app.async_stream_query`, so the same helpers print the
events. The summary table counts the Google searches seen in each answer's events.

In [ ]:
REMOTE_QUESTIONS = [
    "What is the tallest building in the world right now, and how tall is it?",
    "What is the capital of Australia, and roughly how many people live there?",
    "What are the main differences between a hurricane and a tornado?",
]

remote_results = []
for question in REMOTE_QUESTIONS:
    answer, trace = await ask(remote_agent, question)
    show_events("Deployed agent", question, answer, trace)
    searches = sum(row["event"] == "Google Search query" for row in trace)
    remote_results.append((question, bool(answer.strip()), searches))
    await asyncio.sleep(3)  # small pause to stay under rate limits

rows = ["| Question | Answered | Google searches seen in events |", "|---|---|---|"]
for question, answered, searches in remote_results:
    rows.append(f"| {question} | {'yes' if answered else 'NO'} | {searches} |")
display(Markdown("### Summary of deployed-agent tests\n\n" + "\n".join(rows)))

assert all(answered for _, answered, _ in remote_results), "The deployed agent returned an empty answer"
print("Remote test passed: the deployed agent answered every question.")

## 9. Prove it lives in the cloud

A deployed agent is a cloud resource, not a Python variable. Here the notebook forgets `remote_agent` and **re-opens the
deployment from its resource name** (`client.agent_engines.get`), queries it again, and lists the deployments in the project.
After a restart you could do the same with the name saved in `deployed_agent.txt`.

In [ ]:
reconnected = client.agent_engines.get(name=RESOURCE_NAME)
print("Reconnected to:", reconnected.api_resource.display_name)

question = "In one sentence, what is the Agent Development Kit (ADK) from Google?"
answer, trace = await ask(reconnected, question)
show_events("Re-opened deployment", question, answer, trace)
assert answer.strip(), "The re-opened deployment returned an empty answer"

print("\nDeployments in this project and region:")
for count, engine in enumerate(client.agent_engines.list(), start=1):
    print(f"  {engine.api_resource.display_name}  ->  {engine.api_resource.name}")
    if count >= 10:
        print("  ...")
        break

## 10. Cleanup (optional)

A deployment keeps existing, and can keep costing money, until it is deleted. This cell deletes it when
`DELETE_AGENT_WHEN_DONE` is `True`. `force=True` also removes the deployment's managed session data. Leave the flag as
`False` until your work has been reviewed, or delete the deployment from the console later.

In [ ]:
if DELETE_AGENT_WHEN_DONE:
    client.agent_engines.delete(name=RESOURCE_NAME, force=True)
    print("Deleted deployment:", RESOURCE_NAME)
else:
    print("Deployment kept (DELETE_AGENT_WHEN_DONE is False):", RESOURCE_NAME)

## Troubleshooting

| Symptom | Likely cause and fix |
|---|---|
| The bucket cell says there is no access | The account cannot create buckets. Use a bucket listed by the cell (or in the console under Cloud Storage > Buckets, or the lab panel), set `STAGING_BUCKET` to its `gs://` name, and re-run from the config cell. If none exists, ask the instructor |
| `PERMISSION_DENIED` when querying the **deployed** agent (the local test works) | With `identity_type=AGENT_IDENTITY` the agent runs as its own identity, and that identity needs roles that let it call models (the documentation lists Vertex AI Express User, Service Usage Consumer, and Browser). Ask your instructor or an administrator to grant them, following *Use Agent Identity with Agent Runtime*. Alternatively set `USE_AGENT_IDENTITY = False` to use the default service agent, delete the failed deployment (`client.agent_engines.delete(name=RESOURCE_NAME, force=True)`), and run section 7 again |
| `429` / `RESOURCE_EXHAUSTED` | The shared model pool is busy. The `ask` helper retries automatically; if it still fails, wait a few minutes and run the cell again |
| Deployment fails while installing packages | The pinned versions in `requirements` must exist on PyPI. Re-run section 1, restart the session, and deploy again |
| A cell fails at once with `524` / "Could not send execute message to runtime" | This is not a code error: the notebook lost contact with its runtime, usually because the runtime is stuck or overloaded (for example by an install that is still running). Click the stop button, then **Runtime > Restart session**. If that does not help, use the **Connect** menu to disconnect and delete the runtime, reconnect (a new runtime takes a few minutes), and run the install cell again |
| Deployment takes more than 15 minutes | Wait; if the cell finally errors, read the last lines of the message and check Logs Explorer for the deployment |

## Reflection

- What is the difference between running the agent locally (`app`) and the deployed agent (`remote_agent`)? What stays the
  same in the code that calls them?
- Why does the SDK need a Cloud Storage bucket, and what does `requirements` control?
- What does `identity_type=AGENT_IDENTITY` change about who the agent runs as?
- Where do you see in the events that the agent used Google Search?